# Phase 1 -- Madrigal MAPGPS Coverage Audit (Pre-D-144 Evidence)

**Status:** Evidence-gathering only. Running this notebook does NOT constitute Phase 1 acquisition and does NOT approve D-144.

**Governance:** Project Vision Document v4.2 SS1.7 (Recommendation 10), S6.1A, S6.1B, S6.2; Technical Environment and Research Implementation v3.2 S5.1, S7.0 (stage P1-02), EV-23, D-144.

**Why this notebook exists:** the ICTP prepared-VTEC source failed the G-P1A coverage gate (D-143: ARUC 27/365 days, BSHM 35/365 days, NICO 0/365 days, HTTP 404). The Vision document's own Recommendation 10 names MIT Haystack CEDAR Madrigal MAPGPS `gps` binned VTEC as the preferred replacement candidate, but adoption is explicitly conditional: "it becomes the Phase 1 source only after the supervisor approves the map-cell target definition and a target-independent audit confirms adequate 2022 coverage in all three cells, including December." This notebook is that target-independent audit.

**This notebook does NOT:**
- download data for model training
- select or freeze the exact Madrigal instrument/kindat identifier on your behalf -- that is a supervisor freeze gate (Vision SS1.2: "must not be guessed by an implementer or AI coding agent")
- decide pass/fail on the coverage gate -- the numeric coverage minimum is still `TBD -- supervisor freeze gate` per Vision S6.1B

**This notebook DOES:**
1. discover candidate Madrigal instrument + kindat identifiers matching the `gps` binned VTEC product, for you to review and confirm
2. query the confirmed product for calendar year 2022 at the three frozen station-coordinate cells (ARUC, BSHM, NICO)
3. report file-, day-, month- and December-level coverage per station, comparable to the ICTP audit table
4. write immutable evidence artifacts (coverage CSVs, request manifest, SHA-256 hashes) under `/kaggle/working/audit_evidence/`

**Run this on Kaggle with Internet enabled** (Notebook Settings -> Internet -> On). Outputs land under `/kaggle/working` per this project's platform rules.

There are two hard stops built into this notebook (Step A discovery, then a CONFIRMED CONFIGURATION cell). The coverage audit will refuse to run until you have reviewed the discovery output and filled in the confirmed values yourself.


In [ ]:
# Cell 1 -- environment setup
import sys, os, subprocess

print('Python:', sys.version)

try:
    import madrigalWeb.madrigalWeb as madweb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'madrigalWeb'])
    import madrigalWeb.madrigalWeb as madweb

import madrigalWeb
print('madrigalWeb version:', getattr(madrigalWeb, '__version__', 'unknown'))

import json, hashlib, time, socket
from datetime import datetime, timezone
import pandas as pd

OUTPUT_DIR = '/kaggle/working/audit_evidence'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'raw_isprint_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# Internet check -- on Kaggle this requires Settings > Internet: On.
# IMPORTANT: socket.setdefaulttimeout() is GLOBAL and madrigalWeb uses urllib underneath, so a short
# default here would also cap every later isprint() call. A single binned-TEC isprint takes ~3 min of
# server-side extraction (measured), so the DNS probe must restore the previous default when done.
_prev_timeout = socket.getdefaulttimeout()
try:
    socket.setdefaulttimeout(10)
    socket.gethostbyname('cedar.openmadrigal.org')
    print('Internet reachable: cedar.openmadrigal.org resolves OK')
except Exception as exc:
    raise RuntimeError(
        'Cannot resolve cedar.openmadrigal.org. On Kaggle, enable Settings > Internet, '
        'then restart the session and re-run this notebook from the top.'
    ) from exc
finally:
    socket.setdefaulttimeout(_prev_timeout)

# Generous floor for the long isprint calls; None = block indefinitely (urllib default).
socket.setdefaulttimeout(900)
print('socket default timeout set to', socket.getdefaulttimeout(), 's for long isprint calls')


In [ ]:
# Cell 2 -- user identity (CEDAR rules-of-the-road requires a real identity on every request)
# Madrigal logs this identity per request; the project's licensing/citation controls (Tech doc S10.1)
# require it to be traceable to you. Edit here directly if it ever needs to change.
USER_FULLNAME = 'Kimia Rezaei'
USER_EMAIL = 'kiimiiarezaee2025@gmail.com'
USER_AFFILIATION = 'Amirkabir University of Technology'

if 'REPLACE_ME' in USER_FULLNAME or 'REPLACE_ME' in USER_EMAIL:
    raise RuntimeError('Set USER_FULLNAME and USER_EMAIL to your real identity before continuing.')


In [ ]:
# Cell 2b -- connect to a Madrigal instance
# IMPORTANT: MadrigalData() expects the site ROOT url, NOT a '/madrigal' sub-path. Internally it
# fetches that page and regex-searches the HTML for 'accessData.cgi' to locate the cgi endpoint.
# Passing a sub-path gives one of two confusing errors: 'unable to open url' (404) or 'invalid url'
# (page fetched, marker not found). madrigalWeb's own docstring uses: MadrigalData('http://cedar.openmadrigal.org')

MADRIGAL_URL_CANDIDATES = [
    'https://cedar.openmadrigal.org',              # CEDAR -- carries the MIT MAPGPS GPS TEC product
    'http://cedar.openmadrigal.org',               # plain-http fallback if TLS is blocked
]

madData = None
MADRIGAL_URL = None
connection_errors = []

for candidate in MADRIGAL_URL_CANDIDATES:
    try:
        madData = madweb.MadrigalData(candidate)
        MADRIGAL_URL = candidate
        print('Connected to', MADRIGAL_URL)
        break
    except Exception as exc:
        connection_errors.append((candidate, str(exc)))
        print('Could not connect to', candidate, '-', exc)

if madData is None:
    raise RuntimeError(
        'Could not connect to any known Madrigal instance: ' + str(connection_errors) +
        '. On Kaggle, confirm Settings > Internet is On. Note the two distinct madrigalWeb errors: '
        '"unable to open url" means the page 404d, "invalid url" means the page loaded but had no '
        'accessData.cgi marker -- both usually mean a sub-path was passed instead of the site root. '
        'Current site list is at https://cedar.openmadrigal.org ("Other Madrigal sites").'
    )


In [ ]:
# Cell 3 -- frozen station registry (Vision doc S6.2)
# Authoritative source is the official IGS site log; the coordinates below were cross-checked against
# IGS network pages (igs.org / network.igs.org) when this notebook was authored. Treat as PROVISIONAL
# until validated against the official site-log PDF for each station -- the site log ranks above any
# secondary source in the S6.2 evidence hierarchy.

STATIONS = {
    'ARUC': {'lat': 40.286,    'lon': 44.086,    'source': 'IGS network page -- cross-check against site log required'},
    'BSHM': {'lat': 32.778987, 'lon': 35.022987, 'source': 'IGS network page -- cross-check against site log required'},
    'NICO': {'lat': 35.140989, 'lon': 33.396450, 'source': 'IGS network page -- cross-check against site log required'},
}

# Coordinate-to-cell rule (S6.1A/S6.1B: must be frozen and recorded, not guessed).
# DEFAULT convention adopted here: 1deg x 1deg cell identified by its lower-left (floor) corner, i.e.
#   cell = [floor(lat), floor(lat)+1) x [floor(lon), floor(lon)+1)
# CONFIRM this matches the real bin edges Madrigal returns (see the discovery cells below) before
# treating it as frozen.
import math

def cell_bounds(lat, lon):
    lat0 = math.floor(lat)
    lon0 = math.floor(lon)
    return {'lat_min': lat0, 'lat_max': lat0 + 1, 'lon_min': lon0, 'lon_max': lon0 + 1}

for _code, _meta in STATIONS.items():
    _meta['cell'] = cell_bounds(_meta['lat'], _meta['lon'])

pd.DataFrame(STATIONS).T


## Step A -- Discovery (do not skip)

Find the real Madrigal instrument code and kindat code for the `gps` binned VTEC product, and the
real parameter mnemonics available in an actual file. Vision doc S6.1A names kindat candidate 3500
as a *candidate*, not a frozen value -- confirm it here rather than trusting that number blindly.


In [ ]:
# Cell 4 -- discover candidate TEC instruments on this Madrigal site
# NOTE: filtering on the substring 'gps' alone returns NOTHING. The instrument that actually carries
# the MAPGPS binned VTEC product is code 8000, named 'World-wide GNSS Receiver Network' -- 'GNSS',
# not 'GPS'. Match on any of gps / gnss / tec, then read the printed list yourself.
all_instruments = madData.getAllInstruments()
_KEYWORDS = ('gps', 'gnss', 'tec')
gps_instruments = [i for i in all_instruments
                   if any(k in i.name.lower() for k in _KEYWORDS)]

print(len(all_instruments), 'total instruments on this Madrigal site')
print(len(gps_instruments), 'instruments matching', _KEYWORDS, ':')
for inst in gps_instruments:
    print('  code =', inst.code, ' name =', inst.name)

if not gps_instruments:
    raise RuntimeError('No candidate instrument matched -- inspect all_instruments manually.')


In [ ]:
# Cell 5 -- list 2022 experiments for each GPS-named instrument found above
# This can take a little while on first run; there is no local caching for this discovery step.
candidate_experiments = {}
for inst in gps_instruments:
    exps = madData.getExperiments(inst.code, 2022, 1, 1, 0, 0, 0, 2022, 12, 31, 23, 59, 59)
    candidate_experiments[inst.code] = exps
    print('instrument', inst.code, '-', inst.name, ':', len(exps), 'experiments in 2022')


In [ ]:
# Cell 6 -- for each candidate instrument, sample one 2022 experiment and list its files/kindats
# Look for a kindat whose description mentions "binned" and "vtec" (not "line of sight" / "los" --
# per Vision S6.1A the los product is explicitly NOT the Phase 1 replacement candidate).
sample_files_by_instrument = {}
for _icode, exps in candidate_experiments.items():
    if not exps:
        continue
    sample_exp = exps[len(exps) // 2]
    files = madData.getExperimentFiles(sample_exp.id)
    sample_files_by_instrument[_icode] = (sample_exp, files)
    print('instrument', _icode, ' sample experiment:', sample_exp.name, ' id =', sample_exp.id)
    for f in files:
        print('   kindat =', f.kindat, ' desc =', f.kindatdesc, ' file =', f.name)
    print()


In [ ]:
# Cell 7 -- inspect the real parameter mnemonics available in one sample file
# Set FILE_TO_INSPECT to one of the f.name strings printed by Cell 6, then re-run this cell.
# A known-good 2022 kindat-3500 path is pre-filled so this cell is runnable as-is; swap it for any
# filename Cell 6 printed if you want to spot-check a different day.
FILE_TO_INSPECT = '/opt/openmadrigal/madroot/experiments4/2022/gps/15jun22/gps220615g.002.hdf5'

if FILE_TO_INSPECT:
    parms = madData.getExperimentFileParameters(FILE_TO_INSPECT)
    for p in parms:
        print(p.mnemonic, '-', p.description, ' units =', getattr(p, 'units', ''))
else:
    print('Set FILE_TO_INSPECT above to a filename from Cell 6, then re-run this cell.')


## STOP -- freeze gate

The values in the next cell were **measured** against CEDAR Madrigal, not guessed. A discovery run
executed on 2026-08-11 returned:

- instrument `8000` = *World-wide GNSS Receiver Network* (the only global GNSS TEC instrument on the site)
- **366 experiments in calendar 2022** -- one per day, full year, December included
- kindat `3500` = *"TEC binned 1 degree by 1 degree by 5 min"* -- exactly the prepared binned VTEC
  product Vision S6.1A names. Its sibling kindat `3505` is *"Line of sight TEC data"*, the `los`
  product S6.1A explicitly rules out of Phase 1, and `3506` is a site list.
- parameter mnemonics as listed below

One correction worth flagging: the VTEC value parameter is **`tec`** (described as *"Vertically
integrated electron density"*, units TECU), **not** `vtec`. The companion error field is `dtec`.

Re-run cells 4-7 yourself to reproduce this independently -- the numbers above are a starting point,
not a substitute for your own executed evidence.

**Measured is still not approved.** Vision doc SS1.2 requires a freeze gate to be *"resolved,
recorded, and approved"*; this covers resolved and recorded only. D-144 sign-off remains outstanding,
and the coordinate-to-cell convention (lower-left floor, Cell 3) is still an unverified assumption --
sanity-check returned `gdlat`/`glon` values against the station coordinates before freezing it.


In [ ]:
# Cell 8 -- CONFIRMED CONFIGURATION
# Values below were measured from CEDAR Madrigal on 2026-08-11 (see the STOP cell above).
# Re-run cells 4-7 to reproduce independently before relying on them.
INSTRUMENT_CODE = 8000       # World-wide GNSS Receiver Network -- CONFIRMED via getAllInstruments()
KINDAT_CODE = 3500           # 'TEC binned 1 degree by 1 degree by 5 min' -- CONFIRMED (3505 = los, excluded)
PARM_TIME = 'ut1_unix'       # UT1_UNIX -- Unix seconds at interval start
PARM_LAT = 'gdlat'           # GDLAT -- geodetic latitude of measurement, deg
PARM_LON = 'glon'            # GLON -- geographic longitude of measurement, deg
PARM_VTEC = 'tec'            # TEC -- 'Vertically integrated electron density', TECU. NOTE: 'tec', not 'vtec'
PARM_VTEC_ERROR = 'dtec'     # DTEC -- error in vertically integrated electron density, TECU

_required = [INSTRUMENT_CODE, KINDAT_CODE, PARM_TIME, PARM_LAT, PARM_LON, PARM_VTEC]
if any(v is None for v in _required):
    raise RuntimeError(
        'Fill INSTRUMENT_CODE, KINDAT_CODE and the PARM_* mnemonics from Step A before running the '
        'coverage audit. This is a freeze gate -- values must not be guessed.'
    )

PARMS_TO_REQUEST = [PARM_TIME, PARM_LAT, PARM_LON, PARM_VTEC] + ([PARM_VTEC_ERROR] if PARM_VTEC_ERROR else [])
print('Confirmed instrument:', INSTRUMENT_CODE, ' kindat:', KINDAT_CODE)
print('Requesting parameters:', PARMS_TO_REQUEST)


## Step B -- Per-cell 2022 coverage audit

For the confirmed instrument/kindat, walk every 2022 experiment file, filter to each station's frozen
1x1 degree cell, and record which UTC days/hours actually contain a valid (non-fill) VTEC value.
Results are cached to disk under `raw_isprint_cache/` so the notebook can be safely re-run or resumed
on Kaggle without re-fetching everything.


In [ ]:
# Cell 9 -- audit helper functions
#
# COST MODEL (measured against cedar.openmadrigal.org): one isprint() call against a daily binned-TEC
# file costs ~160-180 s regardless of how narrow the lat/lon filter is -- the server pays to read the
# whole global file either way. The original per-station loop therefore paid that cost THREE times per
# day for no extra data. We now issue ONE call per file with a bounding box covering all three cells
# and split the rows locally: same evidence, one third of the wall clock.
#   December only : 31 calls  ~= 1.5 h   (was ~93 calls ~= 4.5 h)
#   Full year 2022: 366 calls ~= 17 h    (was ~1100 calls ~= 50 h -- exceeds Kaggle's 12 h session)
# Run the full year in monthly chunks; the cache below makes each re-run resume for free.

BBOX = {
    'lat_min': min(m['cell']['lat_min'] for m in STATIONS.values()),
    'lat_max': max(m['cell']['lat_max'] for m in STATIONS.values()),
    'lon_min': min(m['cell']['lon_min'] for m in STATIONS.values()),
    'lon_max': max(m['cell']['lon_max'] for m in STATIONS.values()),
}
print('Union bounding box for one-call-per-file fetch:', BBOX)


def cache_path(filename):
    safe_name = filename.replace('/', '_').replace('\\', '_')
    return os.path.join(CACHE_DIR, 'bbox__' + safe_name + '.txt')


def looks_like_isprint_error(text):
    # madrigalWeb does not always raise: some server-side failures come back as an HTML/traceback
    # BODY with a 200 status. Caching such a body would poison the cache permanently, so detect it.
    if not text or not text.strip():
        return True
    head = text.lstrip()[:2000].lower()
    for marker in ('<html', '<!doctype', 'traceback', 'error occurred', 'invalid', 'exception'):
        if marker in head:
            return True
    return False


def fetch_isprint(filename):
    """Fetch every row inside the union bounding box for one experiment file. Cached on disk."""
    filter_string = (
        'filter=' + PARM_LAT + ',' + str(BBOX['lat_min']) + ',' + str(BBOX['lat_max']) +
        ' filter=' + PARM_LON + ',' + str(BBOX['lon_min']) + ',' + str(BBOX['lon_max'])
    )
    parm_string = ','.join(PARMS_TO_REQUEST)
    path = cache_path(filename)
    if os.path.exists(path):
        with open(path, 'r') as fh:
            return fh.read()
    for attempt in range(3):
        try:
            result = madData.isprint(
                filename, parm_string, filter_string,
                USER_FULLNAME, USER_EMAIL, USER_AFFILIATION,
            )
            if looks_like_isprint_error(result):
                raise RuntimeError('server returned an error body, not data: ' + repr(result[:200]))
            with open(path, 'w') as fh:
                fh.write(result)
            time.sleep(0.2)
            return result
        except Exception as exc:
            wait = 2 ** attempt
            print('isprint retry', attempt, 'for', filename, '-', exc, '- waiting', wait, 's')
            time.sleep(wait)
    print('FAILED after retries:', filename)
    return ''


def parse_isprint(text, cell):
    """Post-filter the bounding-box response down to one station cell.

    Independent of the server-side filter on purpose: correctness never depends on the filter syntax.
    Madrigal labels each 1x1 deg bin by an INTEGER gdlat/glon, so the half-open
    [floor, floor+1) test selects exactly one grid label per station (ARUC 40/44, BSHM 32/35,
    NICO 35/33 -- verified against real 2022 output).
    """
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != len(PARMS_TO_REQUEST):
            continue
        try:
            values = [float(p) for p in parts]
        except ValueError:
            continue  # header line or 'missing'/'assumed' fill token
        record = dict(zip(PARMS_TO_REQUEST, values))
        lat_val = record.get(PARM_LAT)
        lon_val = record.get(PARM_LON)
        if lat_val is None or lon_val is None:
            continue
        if not (cell['lat_min'] <= lat_val < cell['lat_max']):
            continue
        if not (cell['lon_min'] <= lon_val < cell['lon_max']):
            continue
        vtec_val = record.get(PARM_VTEC)
        if vtec_val is None or vtec_val < 0 or vtec_val > 1e6:
            continue
        rows.append(record)
    return rows


In [ ]:
# Cell 10 -- run the coverage audit for all three stations across 2022
# RUNTIME (measured, not estimated): ~170 s per experiment file, one file per UTC day, ONE call per
# file for all three stations. December-only ~= 1.5 h cold; a full year ~= 17 h, which does NOT fit a
# single 12 h Kaggle session -- run it in monthly chunks. Every response is cached under
# raw_isprint_cache/, so an interrupted or re-run notebook resumes cheaply.
# START WITH RUN_MONTHS = [12]: December is the locked-test month and the one that matters most.

AUDIT_YEAR = 2022                  # the one governed calendar year (D-8). Never widen this.
RUN_MONTHS = [12]                  # December-only. Widen a few months at a time, not to all 12 at once.

# The window below deliberately spans the whole audit year, so getExperiments returns every
# experiment that OVERLAPS 2022 -- including the 31-Dec-2021 experiment (ends 2022-01-01) and the
# 31-Dec-2022 experiment (ends 2023-01-01). Selecting from that set on month alone is year-blind:
# it admits a 31-December experiment from EITHER year. _RUN_TARGETS makes the test (year, month).
_RUN_TARGETS = {(AUDIT_YEAR, m) for m in RUN_MONTHS}

year_experiments = madData.getExperiments(
    INSTRUMENT_CODE, AUDIT_YEAR, 1, 1, 0, 0, 0, AUDIT_YEAR, 12, 31, 23, 59, 59
)
print(len(year_experiments), 'experiments found for instrument', INSTRUMENT_CODE, 'in 2022')

audit_rows = []
file_errors = []
_files_done = 0

for exp in year_experiments:
    if ((exp.startyear, exp.startmonth) not in _RUN_TARGETS
            and (exp.endyear, exp.endmonth) not in _RUN_TARGETS):
        continue
    try:
        files = madData.getExperimentFiles(exp.id)
    except Exception as exc:
        file_errors.append({'experiment_id': exp.id, 'error': str(exc)})
        continue
    matching_files = [f for f in files if f.kindat == KINDAT_CODE]
    if not matching_files:
        file_errors.append({'experiment_id': exp.id,
                            'error': 'no file with kindat ' + str(KINDAT_CODE)})
    for f in matching_files:
        text = fetch_isprint(f.name)           # ONE server call, all three cells
        if not text:
            file_errors.append({'experiment_id': exp.id, 'file': f.name,
                                'error': 'isprint failed after retries'})
            continue
        for station_code, meta in STATIONS.items():
            rows = parse_isprint(text, meta['cell'])
            for r in rows:
                r['station'] = station_code
                r['experiment_id'] = exp.id
                r['file'] = f.name
            audit_rows.extend(rows)
        _files_done += 1
        print('  [%d] %s -> %d in-cell rows so far' % (_files_done, f.name, len(audit_rows)))

print('Collected', len(audit_rows), 'in-cell VTEC records across all stations')
print(len(file_errors), 'file-level errors (see file_errors)')


In [ ]:
# Cell 11 -- aggregate to day/month coverage per station
#
# YEAR GUARD (do not remove): Madrigal experiment records straddle UTC day boundaries, so a run with
# RUN_MONTHS = [12] legitimately pulls the 31dec21 and 30nov22 files too. Counting their rows as 2022
# December days produced a december_coverage_pct of 103.226% (32 days in a 31-day month) before this
# guard existed. Coverage statistics are computed on calendar-2022 rows ONLY; the raw records CSV
# still keeps every row fetched, out-of-year ones included, so the evidence stays complete.
AUDIT_YEAR = 2022

df_all = pd.DataFrame(audit_rows)

if df_all.empty:
    raise RuntimeError(
        'No in-cell records were parsed. Re-check INSTRUMENT_CODE, KINDAT_CODE, PARM_* mnemonics and '
        'the filter string format in Cell 9 before concluding coverage is zero -- a config mistake and '
        'genuine zero coverage look identical here and must not be confused.'
    )

df_all['timestamp_utc'] = pd.to_datetime(df_all[PARM_TIME], unit='s', utc=True)
df_all['year'] = df_all['timestamp_utc'].dt.year
df_all['date'] = df_all['timestamp_utc'].dt.date
df_all['month'] = df_all['timestamp_utc'].dt.month
df_all['hour'] = df_all['timestamp_utc'].dt.floor('h')

_out_of_year = int((df_all['year'] != AUDIT_YEAR).sum())
print('Fetched rows:', len(df_all), '| out-of-year rows excluded from coverage stats:', _out_of_year)

df = df_all[df_all['year'] == AUDIT_YEAR].copy()
if df.empty:
    raise RuntimeError('Every fetched row fell outside ' + str(AUDIT_YEAR) + ' -- check RUN_MONTHS.')

# 2022 is not a leap year; derive rather than hard-code so the guard survives a year change.
_days_in_year = 366 if (AUDIT_YEAR % 4 == 0 and (AUDIT_YEAR % 100 != 0 or AUDIT_YEAR % 400 == 0)) else 365

summary_rows = []
for station_code in STATIONS:
    sdf = df[df['station'] == station_code]
    dec = sdf[sdf['month'] == 12]
    unique_days = sdf['date'].nunique()
    december_days = dec['date'].nunique()
    summary_rows.append({
        'station': station_code,
        'records_in_cell': len(sdf),
        'unique_days': unique_days,
        'coverage_pct_days': round(100.0 * unique_days / _days_in_year, 3),
        'unique_hourly_bins': sdf['hour'].nunique(),
        'december_days_present': december_days,
        'december_coverage_pct': round(100.0 * december_days / 31.0, 3),
        'months_run': ','.join(str(m) for m in RUN_MONTHS),
    })

summary_df = pd.DataFrame(summary_rows)

# A percentage above 100 means the year guard failed -- fail loudly rather than shipping the number.
_bad = summary_df[summary_df['december_coverage_pct'] > 100.0]
if not _bad.empty:
    raise RuntimeError('december_coverage_pct exceeds 100% -- year guard failed:\n' + str(_bad))

# NOTE: coverage_pct_days is only meaningful once RUN_MONTHS covers the full year. With
# RUN_MONTHS = [12] it reads ~8.5% by construction and is NOT a whole-year coverage figure.
if sorted(RUN_MONTHS) != list(range(1, 13)):
    print('PARTIAL RUN -- RUN_MONTHS =', RUN_MONTHS,
          '-- coverage_pct_days is a partial-year figure, not the G-P1A whole-year number.')

summary_df


In [ ]:
# Cell 12 -- monthly breakdown per station -- needed to judge F1-F4 fold and December-test viability
# Built from df (calendar-AUDIT_YEAR rows only), so a straddling 31dec21 / 30nov22 file cannot leak in.
monthly = (
    df.groupby(['station', 'month'])['date']
      .nunique()
      .rename('days_with_data')
      .reset_index()
)
monthly_pivot = monthly.pivot(index='month', columns='station', values='days_with_data').fillna(0).astype(int)
monthly_pivot


In [ ]:
# Cell 13 -- write immutable evidence artifacts with hashes (NFR-AUD-01 / P1-02 output contract)

def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

summary_csv_path = os.path.join(OUTPUT_DIR, 'madrigal_coverage_summary.csv')
monthly_csv_path = os.path.join(OUTPUT_DIR, 'madrigal_coverage_monthly.csv')
raw_records_path = os.path.join(OUTPUT_DIR, 'madrigal_coverage_raw_records.csv')

summary_df.to_csv(summary_csv_path, index=False)
monthly_pivot.to_csv(monthly_csv_path)
df_all.to_csv(raw_records_path, index=False)   # every fetched row, out-of-year included

manifest = {
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
    'madrigal_url': MADRIGAL_URL,
    'madrigalWeb_version': getattr(madrigalWeb, '__version__', 'unknown'),
    'instrument_code': INSTRUMENT_CODE,
    'kindat_code': KINDAT_CODE,
    'parameters_requested': PARMS_TO_REQUEST,
    'stations': STATIONS,
    'coordinate_to_cell_convention': 'lower-left floor, 1deg x 1deg -- confirm against real bin edges before freezing',
    'audit_year': AUDIT_YEAR,
    'run_months': RUN_MONTHS,
    'partial_run': sorted(RUN_MONTHS) != list(range(1, 13)),
    'rows_fetched_total': int(len(df_all)),
    'rows_outside_audit_year_excluded': int((df_all['year'] != AUDIT_YEAR).sum()),
    'file_level_errors': file_errors,
    'user_fullname': USER_FULLNAME,
    'user_affiliation': USER_AFFILIATION,
}

manifest_path = os.path.join(OUTPUT_DIR, 'request_manifest.json')
with open(manifest_path, 'w') as fh:
    json.dump(manifest, fh, indent=2, default=str)

hashes = {}
for path in [summary_csv_path, monthly_csv_path, raw_records_path, manifest_path]:
    hashes[os.path.basename(path)] = sha256_of_file(path)

hashes_path = os.path.join(OUTPUT_DIR, 'sha256_manifest.json')
with open(hashes_path, 'w') as fh:
    json.dump(hashes, fh, indent=2)

print('Evidence written to', OUTPUT_DIR)
for name, digest in hashes.items():
    print(' ', name, digest)


## Result -- this is evidence, not a decision

Compare `madrigal_coverage_summary.csv` against the ICTP audit table for calibration:

| Station | ICTP (rejected, D-143) | Madrigal MAPGPS `gps` (this audit) |
|---|---|---|
| ARUC | 27/365 days (7.397%) | see `summary_df` above |
| BSHM | 35/365 days (9.589%) | see `summary_df` above |
| NICO | 0/365 days (0.000%, HTTP 404) | see `summary_df` above |

Required for a pass, per Vision doc S6.1B (numeric minimum still `TBD -- supervisor freeze gate`):
- readable 2022 data in all three cells
- adequate common-date coverage across F1-F4 (April/July/October/November validation months) and December
- no unexplained product discontinuity
- compliant citation/acknowledgment -- record the permanent experiment citation from the Madrigal
  web interface for every experiment used, per CEDAR rules-of-the-road (this notebook does not fetch
  citations automatically; the API surface for that varies by Madrigal site version, so pulling it
  programmatically here would risk quietly using a wrong or missing citation string)

**Do not treat a good coverage number here as automatic approval.** D-144 (replacement-source
decision) and the G-P1A gate both require explicit student + supervisor sign-off, and this project's
TEC governance overlay requires a human/supervisor read of the evidence before any stage advances.
Bring `request_manifest.json`, `sha256_manifest.json`, `madrigal_coverage_summary.csv` and
`madrigal_coverage_monthly.csv` to that review.
